In [1]:
using Plots
using Statistics
using Random
using Printf

# Load code từ project
include("../src/utils.jl")
include("../src/algorithm/declat.jl")

mkpath("../results/csv")
mkpath("../results/charts")

println("Environment ready")

Environment ready


In [2]:
datasets = Dict(
    "chess" => "../data/benchmark/chess.txt",
    "mushroom" => "../data/benchmark/mushroom.txt",
    "retail" => "../data/benchmark/retail.txt"
)

minsups_map = Dict(
    "chess" => [2000,1500],
    "mushroom" => [4000,3000],
    "retail" => [3000,2000]
)

Dict{String, Vector{Int64}} with 3 entries:
  "retail"   => [3000, 2000]
  "chess"    => [2000, 1500]
  "mushroom" => [4000, 3000]

In [3]:
function load_dataset(name, path)
    println("Loading $name ...")
    return read_spmf(path)
end

datasets_data = Dict()

for (name,path) in datasets
    datasets_data[name] = load_dataset(name,path)
end

println("Datasets loaded")

Loading retail ...
Loading chess ...
Loading mushroom ...
Datasets loaded


In [4]:
function benchmark_runtime(transactions, minsups; mode=:optimized)

    runtimes = Float64[]
    counts = Int[]

    for ms in minsups

        t = @elapsed res = run_declat(transactions, ms; mode=mode)

        push!(runtimes, t * 1000)
        push!(counts, length(res))

    end

    return runtimes, counts
end


function result_pairs(res)
    Set((Tuple(x.items), x.support) for x in res)
end


function compare_correctness(myres, ref)

    myset = result_pairs(myres)
    refset = result_pairs(ref)

    match = length(intersect(myset,refset))
    total = length(refset)

    ratio = total == 0 ? 1.0 : match / total

    return match,total,ratio
end

compare_correctness (generic function with 1 method)

In [5]:
GC.gc()

println("Running correctness test...")

correctness_rows = []

for (name,tx) in datasets_data
    
    ms = minsups_map[name][end]

    println("Testing $name (minsup=$ms)")

    opt = run_declat(tx, ms; mode=:optimized)
    base = run_declat(tx, ms; mode=:baseline)

    match,total,ratio = compare_correctness(opt,base)

    push!(correctness_rows,(
        dataset=name,
        minsup=ms,
        optimized_count=length(opt),
        baseline_count=length(base),
        matched=match,
        ratio=ratio
    ))

    GC.gc()
end

Running correctness test...
Testing retail (minsup=2000)
Testing chess (minsup=1500)
Testing mushroom (minsup=3000)


In [6]:
open("../results/csv/correctness.csv","w") do io

    println(io,"dataset,minsup,optimized_count,baseline_count,matched,ratio")

    for r in correctness_rows

        println(io,
        "$(r.dataset),$(r.minsup),$(r.optimized_count),$(r.baseline_count),$(r.matched),$(r.ratio)"
        )

    end

end

println("Saved correctness.csv")

Saved correctness.csv


In [7]:
GC.gc()

println("Running runtime benchmark...")

runtime_rows = []

for (name,tx) in datasets_data

    minsups = minsups_map[name]

    println("Benchmarking $name")

    runtimes_opt,counts_opt = benchmark_runtime(tx,minsups;mode=:optimized)
    runtimes_base,counts_base = benchmark_runtime(tx,minsups;mode=:baseline)

    for i in eachindex(minsups)

        push!(runtime_rows,(
            dataset=name,
            minsup=minsups[i],
            optimized_ms=runtimes_opt[i],
            baseline_ms=runtimes_base[i],
            optimized_count=counts_opt[i],
            baseline_count=counts_base[i]
        ))

    end

    p = plot(
        minsups,
        runtimes_base,
        marker=:circle,
        label="Baseline",
        xlabel="minsup",
        ylabel="Time (ms)",
        title="Runtime vs minsup - $name",
        xflip=true
    )

    plot!(p,minsups,runtimes_opt,marker=:square,label="Optimized")

    savefig(p,"../results/charts/runtime_$name.png")

    GC.gc()

end

Running runtime benchmark...
Benchmarking retail
Benchmarking chess
Benchmarking mushroom


In [8]:
open("../results/csv/runtime_vs_minsup.csv","w") do io

    println(io,"dataset,minsup,optimized_ms,baseline_ms,optimized_count,baseline_count")

    for r in runtime_rows

        println(io,
        "$(r.dataset),$(r.minsup),$(r.optimized_ms),$(r.baseline_ms),$(r.optimized_count),$(r.baseline_count)"
        )

    end

end

println("Saved runtime_vs_minsup.csv")

Saved runtime_vs_minsup.csv


In [9]:
GC.gc()

println("Generating itemset charts...")

for (name,tx) in datasets_data

    minsups = minsups_map[name]

    _,counts = benchmark_runtime(tx,minsups;mode=:optimized)

    p = plot(
        minsups,
        counts,
        marker=:diamond,
        label="Optimized",
        xlabel="minsup",
        ylabel="Frequent itemsets",
        title="Output size vs minsup - $name",
        xflip=true
    )

    savefig(p,"../results/charts/itemsets_$name.png")

    GC.gc()

end

Generating itemset charts...
